# Audit Data Awal — Dataset Crime Incidents

**Dataset:** `crime_incidents_messy.csv`
**Tujuan:** Melakukan audit awal terhadap dataset untuk mengidentifikasi jumlah data, jumlah kolom, missing value, duplikat, tipe data, dan permasalahan kualitas data lainnya sebelum proses cleaning dilakukan.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('crime_incidents_messy.csv')
print("Dataset berhasil dimuat.")


Dataset berhasil dimuat.


## 1. Jumlah Data dan Jumlah Kolom

In [2]:
jumlah_baris, jumlah_kolom = df.shape
print(f"Jumlah data (baris): {jumlah_baris}")
print(f"Jumlah kolom       : {jumlah_kolom}")


Jumlah data (baris): 5250
Jumlah kolom       : 33


In [3]:
print("Daftar kolom:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")


Daftar kolom:
 1. incident_id
 2. crime_type
 3. district
 4. city
 5. state
 6. address
 7. latitude
 8. longitude
 9. incident_datetime
10. officer_id
11. officer_first_name
12. officer_last_name
13. badge_number
14. suspect_id
15. suspect_first_name
16. suspect_last_name
17. suspect_age
18. suspect_gender
19. suspect_race
20. victim_id
21. victim_first_name
22. victim_last_name
23. victim_age
24. victim_gender
25. victim_phone
26. weapon_used
27. severity
28. case_status
29. resolution
30. num_arrests
31. property_loss_usd
32. reported_online
33. notes


## 2. Tipe Data per Kolom

In [4]:
print(df.dtypes)


incident_id               str
crime_type                str
district                  str
city                      str
state                     str
address                   str
latitude              float64
longitude             float64
incident_datetime         str
officer_id                str
officer_first_name        str
officer_last_name         str
badge_number          float64
suspect_id                str
suspect_first_name        str
suspect_last_name         str
suspect_age           float64
suspect_gender            str
suspect_race              str
victim_id                 str
victim_first_name         str
victim_last_name          str
victim_age            float64
victim_gender             str
victim_phone              str
weapon_used               str
severity                  str
case_status               str
resolution                str
num_arrests           float64
property_loss_usd         str
reported_online           str
notes                     str
dtype: obj

**Catatan:** Beberapa kolom yang seharusnya numerik (misalnya `property_loss_usd`) masih terbaca sebagai `object`/`string`, kemungkinan karena adanya karakter atau format yang tidak konsisten pada data mentah.

## 3. Missing Value per Kolom

In [5]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({
    'jumlah_missing': missing,
    'persentase (%)': missing_pct
}).sort_values('jumlah_missing', ascending=False)

print(missing_summary[missing_summary['jumlah_missing'] > 0])


                    jumlah_missing  persentase (%)
suspect_race                  1601           30.50
suspect_gender                1413           26.91
suspect_age                   1097           20.90
notes                         1069           20.36
victim_phone                  1056           20.11
victim_gender                 1000           19.05
weapon_used                    970           18.48
resolution                     951           18.11
suspect_last_name              810           15.43
suspect_id                     810           15.43
suspect_first_name             810           15.43
case_status                    731           13.92
victim_age                     562           10.70
reported_online                504            9.60
property_loss_usd              436            8.30
severity                       355            6.76
incident_datetime              340            6.48
num_arrests                    333            6.34
badge_number                   

In [6]:
total_missing = df.isnull().sum().sum()
total_cell = df.shape[0] * df.shape[1]
print(f"Total sel kosong   : {total_missing}")
print(f"Total sel dataset  : {total_cell}")
print(f"Persentase missing : {round(total_missing/total_cell*100, 2)}%")
print(f"Jumlah kolom yang memiliki missing value: {(missing > 0).sum()} dari {df.shape[1]} kolom")


Total sel kosong   : 16478
Total sel dataset  : 173250
Persentase missing : 9.51%
Jumlah kolom yang memiliki missing value: 24 dari 33 kolom


## 4. Pemeriksaan Data Duplikat

In [7]:
dup_full = df.duplicated().sum()
print(f"Jumlah baris duplikat penuh (semua kolom identik): {dup_full}")

dup_id = df['incident_id'].duplicated().sum()
print(f"Jumlah incident_id duplikat: {dup_id}")


Jumlah baris duplikat penuh (semua kolom identik): 200
Jumlah incident_id duplikat: 200


In [8]:
# Menampilkan contoh baris duplikat
df[df.duplicated(keep=False)].sort_values('incident_id').head(10)


## 5. Permasalahan yang Ditemukan

### 5.1 Inkonsistensi Kategori pada Kolom Teks

Beberapa kolom kategori memiliki variasi penulisan (huruf besar/kecil, spasi berlebih, singkatan berbeda) untuk nilai yang seharusnya sama.

In [9]:
print("Jumlah nilai unik mentah pada 'crime_type':", df['crime_type'].nunique())

crime_type_clean = df['crime_type'].astype(str).str.strip().str.lower()
print("Setelah normalisasi huruf kecil & strip spasi:", crime_type_clean.nunique(), "kategori unik")
print()
print("Contoh isi kolom crime_type (nilai mentah, unik, 15 pertama):")
print(sorted(df['crime_type'].dropna().unique())[:15])


Jumlah nilai unik mentah pada 'crime_type': 182
Setelah normalisasi huruf kecil & strip spasi: 68 kategori unik

Contoh isi kolom crime_type (nilai mentah, unik, 15 pertama):
['  ASSAULT  ', '  Abduction  ', '  B&E  ', '  Battery  ', '  Cybercrime  ', '  D.U.I.  ', '  DRUG OFFENSE  ', '  DUI  ', '  DUII  ', '  DV  ', '  DWI  ', '  Dom. Violence  ', '  Drug Offence  ', '  FRAUD  ', '  Fraudulent Activity  ']


In [10]:
print("Jumlah nilai unik mentah pada 'district':", df['district'].nunique())

district_clean = df['district'].astype(str).str.strip().str.lower()
print("Setelah normalisasi huruf kecil & strip spasi:", district_clean.nunique(), "kategori unik")
print()
print("Distribusi district setelah normalisasi (masih ada singkatan seperti 'nor', 'sou', 'mid'):")
print(district_clean.value_counts())


Jumlah nilai unik mentah pada 'district': 131
Setelah normalisasi huruf kecil & strip spasi: 16 kategori unik

Distribusi district setelah normalisasi (masih ada singkatan seperti 'nor', 'sou', 'mid'):
district
southwest    465
west         462
northeast    449
central      446
midtown      438
north        434
northwest    430
south        426
southeast    399
east         393
nor          283
sou          270
mid           97
cen           96
wes           82
eas           80
Name: count, dtype: int64


In [11]:
for col in ['severity', 'case_status', 'resolution', 'reported_online', 'weapon_used',
            'suspect_gender', 'victim_gender', 'suspect_race']:
    print(f"--- {col} ---")
    print(sorted(df[col].dropna().unique()))
    print()


--- severity ---
['1', '2', '3', '4', 'CRITICAL', 'Crit', 'Critical', 'High', 'Low', 'MEDIUM', 'Med', 'Medium', 'high', 'low']

--- case_status ---
['CLOSED', 'Closed', 'Investgation', 'OPEN', 'Open', 'Pending', 'Pendng', 'Resolved', 'Under Investigation', 'closed', 'open', 'under investigation']

--- resolution ---
['Arres Made', 'Arrest Made', 'Case Dismissed', 'Dismissed', 'NO ARREST', 'No Arrest', 'Warning Issued', 'arrest made', 'warning']

--- reported_online ---
['0', '1', 'False', 'NO', 'No', 'True', 'YES', 'Yes', 'no', 'yes']

--- weapon_used ---
['Bat', 'Blunt Object', 'Firearm', 'Gun', 'Hands/Feet', 'KNIFE', 'Knife', 'Pistol', 'Rifle', 'Unarmed', 'blunt object', 'firearm', 'hands']

--- suspect_gender ---
['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'Other', 'Unknown', 'f', 'female', 'm', 'male']

--- victim_gender ---
['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'Other', 'Unknown', 'f', 'female', 'm', 'male']

--- suspect_race ---
['Asian', 'BLACK', 'Black', 'Hispanic',

### 5.2 Nilai Numerik Tidak Valid (Outlier / Data Tidak Logis)

In [12]:
neg_suspect_age = (df['suspect_age'] < 0).sum()
high_suspect_age = (df['suspect_age'] > 120).sum()
neg_victim_age = (df['victim_age'] < 0).sum()
high_victim_age = (df['victim_age'] > 120).sum()
neg_arrests = (df['num_arrests'] < 0).sum()

print(f"suspect_age bernilai negatif : {neg_suspect_age}")
print(f"suspect_age > 120 tahun      : {high_suspect_age}")
print(f"victim_age bernilai negatif  : {neg_victim_age}")
print(f"victim_age > 120 tahun       : {high_victim_age}")
print(f"num_arrests bernilai negatif : {neg_arrests}")


suspect_age bernilai negatif : 173
suspect_age > 120 tahun      : 160
victim_age bernilai negatif  : 223
victim_age > 120 tahun       : 185
num_arrests bernilai negatif : 184


In [13]:
loss_numeric = pd.to_numeric(df['property_loss_usd'], errors='coerce')
neg_loss = (loss_numeric < 0).sum()
non_numeric_loss = df['property_loss_usd'].notna().sum() - loss_numeric.notna().sum()

print(f"property_loss_usd bernilai negatif        : {neg_loss}")
print(f"property_loss_usd berformat non-numerik   : {non_numeric_loss}")


property_loss_usd bernilai negatif        : 206
property_loss_usd berformat non-numerik   : 139


In [14]:
invalid_lat = ((df['latitude'] < -90) | (df['latitude'] > 90)).sum()
invalid_lon = ((df['longitude'] < -180) | (df['longitude'] > 180)).sum()

print(f"latitude di luar rentang valid (-90 s.d. 90)   : {invalid_lat}")
print(f"longitude di luar rentang valid (-180 s.d. 180): {invalid_lon}")


latitude di luar rentang valid (-90 s.d. 90)   : 184
longitude di luar rentang valid (-180 s.d. 180): 0


### 5.3 Format Tanggal Tidak Konsisten pada `incident_datetime`

In [15]:
import re

def klasifikasi_format(x):
    if pd.isna(x):
        return 'missing'
    x = str(x).strip()
    if re.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$', x):
        return 'YYYY-MM-DD HH:MM:SS'
    if re.match(r'^\d{2}-\d{2}-\d{4}$', x):
        return 'DD-MM-YYYY'
    if re.match(r'^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$', x):
        return 'MM/DD/YYYY HH:MM'
    return 'format lain'

format_counts = df['incident_datetime'].apply(klasifikasi_format).value_counts()
print(format_counts)


incident_datetime
YYYY-MM-DD HH:MM:SS    4530
missing                 340
MM/DD/YYYY HH:MM        218
DD-MM-YYYY              162
Name: count, dtype: int64


## 6. Ringkasan Permasalahan yang Ditemukan

1. **Missing value** ditemukan pada 24 dari 33 kolom, dengan `suspect_race` (30,50%) dan `suspect_gender` (26,91%) sebagai kolom dengan proporsi missing tertinggi.
2. **200 baris duplikat penuh** (berdasarkan `incident_id` dan seluruh kolom lainnya) yang perlu dihapus.
3. **Inkonsistensi kategori teks** pada hampir semua kolom kategorikal:
   - `crime_type`: 182 nilai unik mentah, masih tersisa 68 kategori setelah dinormalisasi huruf kecil (karena adanya singkatan berbeda seperti "B&E" vs "Breaking & Entering", "DV" vs "Domestic Violence", typo seperti "Homocide", "Arsen").
   - `district`: 131 nilai unik mentah, masih tersisa 16 kategori setelah normalisasi (karena singkatan seperti "nor", "sou", "mid", "cen", "wes").
   - `severity`, `case_status`, `resolution`, `reported_online`, `weapon_used`, `suspect_gender`, `victim_gender`, `suspect_race` seluruhnya memiliki variasi penulisan huruf besar/kecil dan representasi berbeda untuk makna yang sama (contoh: `reported_online` memiliki representasi `True/False`, `Yes/No`, dan `1/0` untuk nilai boolean yang sama).
4. **Nilai numerik tidak logis**:
   - `suspect_age` dan `victim_age` memiliki nilai negatif dan nilai di atas 120 tahun.
   - `num_arrests` memiliki nilai negatif.
   - `property_loss_usd` memiliki nilai negatif dan sebagian tersimpan sebagai teks bercampur dengan format tidak standar.
   - `latitude` memiliki nilai di luar rentang koordinat valid (-90 s.d. 90).
5. **Format tanggal tidak konsisten** pada kolom `incident_datetime`: campuran format `YYYY-MM-DD HH:MM:SS`, `DD-MM-YYYY`, dan `MM/DD/YYYY HH:MM` dalam satu kolom yang sama.

## 7. Kesimpulan

Dataset `crime_incidents_messy.csv` memerlukan proses **data cleaning menyeluruh** sebelum dapat digunakan untuk analisis lebih lanjut. Prioritas utama tahap cleaning berikutnya meliputi: (1) penghapusan baris duplikat, (2) standardisasi kategori teks (case folding, pemetaan singkatan ke kategori baku), (3) penanganan missing value sesuai jenis data (imputasi atau penghapusan), (4) validasi dan koreksi nilai numerik yang tidak logis (usia, jumlah penangkapan, kerugian properti, koordinat), serta (5) penyeragaman format tanggal pada kolom `incident_datetime`.
